In [2]:
!pip install thefuzz[speedup]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.4 MB/s eta 0:00:00


## 1. Carga de la Base Táctica (StatsBomb)

Iniciamos el proceso de enriquecimiento cargando nuestra base de datos maestra generada en el Script 1. Este archivo contiene los perfiles tácticos de los jugadores filtrados por un mínimo de 900 minutos jugados en las 4 grandes ligas europeas durante la temporada 15/16.

In [3]:
import pandas as pd
from IPython.display import display

print("="*85)
print(" 📂 PASO 1: LECTURA DE LA BASE TÁCTICA ORIGINAL")
print("="*85)

# Definir la ruta del archivo táctico (StatsBomb)
ruta_tactica = '/content/drive/MyDrive/dataset_perfiles_jugadores_1516.csv'

# Cargar los datos a memoria
df_tactico = pd.read_csv(ruta_tactica)

# 🚨 PARCHE DE SEGURIDAD: Limpiar columnas financieras previas si existen
columnas_a_limpiar = ['value_eur', 'wage_eur', 'age', 'weight_kg', 'height_cm']
columnas_existentes = [col for col in columnas_a_limpiar if col in df_tactico.columns]

if columnas_existentes:
    df_tactico = df_tactico.drop(columns=columnas_existentes)
    print(f"🧹 Limpieza previa: Se eliminaron columnas financieras antiguas {columnas_existentes}")

print(f"✅ Base Táctica cargada: {df_tactico.shape[0]} jugadores con {df_tactico.shape[1]} atributos.")
display(df_tactico.head(3))

 📂 PASO 1: LECTURA DE LA BASE TÁCTICA ORIGINAL
🧹 Limpieza previa: Se eliminaron columnas financieras antiguas ['value_eur', 'wage_eur', 'age', 'weight_kg', 'height_cm']
✅ Base Táctica cargada: 1699 jugadores con 22 atributos.


,player,team,competition_name,position_group,position_detail,minutos_jugados,xG P90,xG por Tiro,Asistencias a Tiro P90,% Centralidad Equipo,...,% Pases Seguridad,% Pases Bajo Presión,Intercepciones PAdj P90,Tackles Ganados PAdj P90,Duelos Aéreos Ganados P90,Recup. Último Tercio P90,Conducciones Progresivas P90,Influencia Global P90,Faltas Recibidas P90,Pérdidas de Balón P90
0,Aaron Cresswell,West Ham United,Premier League,Defensa,Left Back,3398,0.021197,0.029640,1.085933,11.456168,...,34.845251,3.653949,0.974910,0.615733,0.0,0.979988,2.834020,147.872278,0.927016,1.430253
1,Aaron Lennon,Everton,Premier League,Delantero,Right Wing,2130,0.082510,0.108485,0.887324,2.772703,...,51.744186,3.852740,0.615107,0.615107,0.0,1.225352,2.915493,98.704225,1.098592,1.732394
2,Aaron Ramsey,Arsenal,Premier League,Mediocampista,Left Defensive Midfield,2826,0.234788,0.110035,1.082803,9.927251,...,38.279472,3.962241,1.199566,1.986781,0.0,2.229299,5.477707,256.401274,1.815287,3.949045


## 2. Extracción de Datos Financieros y Demográficos (FIFA 16)

Para añadir la capa financiera (*Moneyball*) y física a nuestro análisis, importamos el dataset de EA Sports FIFA 16. Esta base actúa como un estimador validado y estandarizado del valor de mercado (`value_eur`), salario (`wage_eur`) y métricas físicas (`age`, `height_cm`, `weight_kg`) para la misma temporada. Filtramos estrictamente estas variables para no sobrecargar el modelo.

In [4]:
print("="*85)
print(" 💱 PASO 2: LECTURA Y FILTRADO DE LA BASE FINANCIERA")
print("="*85)

# Definir la ruta del archivo financiero (Kaggle / EA Sports)
ruta_fifa = '/content/drive/MyDrive/players_16.csv'

# Cargar la base de datos
df_fifa = pd.read_csv(ruta_fifa)

# Filtrar solo las columnas analíticas de interés
columnas_fifa = ['long_name', 'value_eur', 'wage_eur', 'age', 'weight_kg', 'height_cm']
df_fifa_filtrado = df_fifa[columnas_fifa].copy()

# Eliminar posibles jugadores duplicados en la base de FIFA (ej. transferencias de invierno)
df_fifa_filtrado = df_fifa_filtrado.drop_duplicates(subset=['long_name'])

print(f"✅ Base Financiera cargada: {df_fifa_filtrado.shape[0]} registros únicos.")
display(df_fifa_filtrado.head(3))

 💱 PASO 2: LECTURA Y FILTRADO DE LA BASE FINANCIERA
✅ Base Financiera cargada: 15599 registros únicos.


/tmp/ipykernel_2496/3845942550.py:9: DtypeWarning: Columns (104) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fifa = pd.read_csv(ruta_fifa)


,long_name,value_eur,wage_eur,age,weight_kg,height_cm
0,Lionel Andrés Messi Cuccittini,111000000.0,550000.0,28,72,170
1,Cristiano Ronaldo dos Santos Aveiro,85500000.0,475000.0,30,80,185
2,Arjen Robben,56000000.0,250000.0,31,80,180


## 3. Cruce Inteligente (Fuzzy Matching) y Auditoría

La integración de ambas bases presenta un desafío clásico en la ingeniería de datos deportivos: la inconsistencia en el registro de los nombres (nombres legales vs. nombres comerciales).

Para solucionar esto sin generar valores nulos masivos, implementamos un algoritmo de coincidencia difusa (*Fuzzy Matching*) utilizando la distancia de Levenshtein. El motor evalúa la similitud de los strings y fusiona los registros si existe al menos un 80% de coincidencia. Finalmente, realizamos una auditoría para validar la tasa de éxito del cruce.

In [8]:
from thefuzz import process

print("="*85)
print(" 🔍 PASO 3: MOTOR DE CRUCE DIFUSO (FUZZY MATCHING)")
print("="*85)

# Extraemos la lista de nombres de FIFA para el motor de búsqueda
nombres_fifa = df_fifa_filtrado['long_name'].tolist()

def buscar_gemelo_nombre(nombre_statsbomb):
    """Busca el nombre más parecido en la base de FIFA. Retorna el nombre si la similitud >= 80%."""
    mejor_coincidencia, similitud = process.extractOne(nombre_statsbomb, nombres_fifa)
    if similitud >= 75:
        return mejor_coincidencia
    return None

print("⏳ Emparejando jugadores (esto tomará aproximadamente 1-2 minutos)...")
df_tactico['nombre_llave_fifa'] = df_tactico['player'].apply(buscar_gemelo_nombre)

# Cruzamos (Merge) ambas bases usando la llave generada
df_master = pd.merge(
    df_tactico,
    df_fifa_filtrado,
    left_on='nombre_llave_fifa',
    right_on='long_name',
    how='left'
)

# Limpieza: Borramos las columnas puente que ya no necesitamos
df_master = df_master.drop(columns=['nombre_llave_fifa', 'long_name'])

# --- AUDITORÍA DE ÉXITO ---
jugadores_cruzados = df_master['value_eur'].notnull().sum()
jugadores_huerfanos = df_master['value_eur'].isnull().sum()
tasa_exito = (jugadores_cruzados / len(df_master)) * 100

print("\n📊 RESULTADOS DE LA AUDITORÍA DE CRUCE:")
print("-" * 85)
print(f" 🟢 Con precio asignado con éxito: {jugadores_cruzados} ({tasa_exito:.1f}%)")
print(f" 🔴 Sin coincidencias (Huérfanos) : {jugadores_huerfanos}")
print("-" * 85)

print("\n✅ Cruce finalizado en memoria. (Base de datos pendiente de guardado)")

# Vista previa de la tabla final
display(df_master[['player', 'team', 'value_eur', 'wage_eur', 'age', 'height_cm']].head())

 🔍 PASO 3: MOTOR DE CRUCE DIFUSO (FUZZY MATCHING)
⏳ Emparejando jugadores (esto tomará aproximadamente 1-2 minutos)...

📊 RESULTADOS DE LA AUDITORÍA DE CRUCE:
-------------------------------------------------------------------------------------
 🟢 Con precio asignado con éxito: 1661 (97.8%)
 🔴 Sin coincidencias (Huérfanos) : 38
-------------------------------------------------------------------------------------

✅ Cruce finalizado en memoria. (Base de datos pendiente de guardado)


,player,team,value_eur,wage_eur,age,height_cm
0,Aaron Cresswell,West Ham United,3000000.0,35000.0,25.0,170.0
1,Aaron Lennon,Everton,625000.0,8000.0,29.0,180.0
2,Aaron Ramsey,Arsenal,26000000.0,130000.0,24.0,177.0
3,Abdelaziz Barrada,Marseille,4300000.0,60000.0,26.0,179.0
4,Abdelaziz Barrada,Olympique de Marseille,4300000.0,60000.0,26.0,179.0


In [6]:
print("="*85)
print(" 🕵️‍♂️ INSPECCIÓN DE JUGADORES NO EMPAREJADOS (HUÉRFANOS)")
print("="*85)

# 1. Aislar a los jugadores que no hicieron match (value_eur es nulo)
df_huerfanos = df_master[df_master['value_eur'].isnull()].copy()

# 2. Definir qué queremos ver: Contexto + Estadísticas Tácticas de Rendimiento
# Puedes agregar o quitar columnas métricas en esta lista según lo que quieras revisar
columnas_inspeccion = [
    'player', 'team', 'position_group', 'minutos_jugados',
    'xG P90', 'Asistencias a Tiro P90', 'Pases Último Tercio P90',
    'Influencia Global P90'
]

# 3. Ordenarlos por minutos jugados para priorizar a los más importantes
df_huerfanos = df_huerfanos.sort_values(by='minutos_jugados', ascending=False)

print(f"⚠️ Cantidad total de jugadores huérfanos: {df_huerfanos.shape[0]}")
print("Mostrando el Top 15 con más minutos jugados y su rendimiento táctico:")
print("-" * 85)

# 4. Mostrar la tabla
display(df_huerfanos[columnas_inspeccion].head(41))

 🕵️‍♂️ INSPECCIÓN DE JUGADORES NO EMPAREJADOS (HUÉRFANOS)
⚠️ Cantidad total de jugadores huérfanos: 41
Mostrando el Top 15 con más minutos jugados y su rendimiento táctico:
-------------------------------------------------------------------------------------


,player,team,position_group,minutos_jugados,xG P90,Asistencias a Tiro P90,Pases Último Tercio P90,Influencia Global P90
1605,Vitorino Hilton da Silva,Montpellier,Defensa,3235,0.050900,0.194745,8.902628,137.323029
1350,Ricardo Alberto Silveira de Carvalho,AS Monaco,Defensa,2931,0.019561,0.092119,4.206755,127.400205
789,Joleon Lescott,West Bromwich Albion,Defensa,2889,0.000000,0.000000,0.716511,8.317757
788,Joleon Lescott,Aston Villa,Defensa,2889,0.048007,0.124611,4.766355,93.551402
371,David Ducourtioux,Gazélec Ajaccio,Mediocampista,2861,0.029134,0.849353,19.912618,173.897239
1593,Vid Belec,Carpi,Portero,2747,0.000000,0.000000,4.947215,64.248271
594,Gianluigi Donnarumma,AC Milan,Portero,2697,0.000000,0.000000,2.369299,76.384872
414,Dieumerci Mbokani Bezua,Norwich City,Delantero,2626,0.206468,0.719726,9.322163,96.134806
1484,Shinji Okazaki,Leicester City,Delantero,2602,0.246354,0.588009,6.779400,115.768640
893,Juraj Kucka,Genoa,Mediocampista,2553,0.000000,0.000000,0.528790,6.415981


In [7]:
print("="*85)
print(" 💾 PASO 4: GUARDAR BASE DE DATOS ENRIQUECIDA")
print("="*85)

# Definir la nueva ruta de salida (así no sobrescribimos los datos crudos del Script 1)
ruta_salida_enriquecida = '/content/drive/MyDrive/dataset_master_enriquecido_1516.csv'

# Exportar el DataFrame a CSV
df_master.to_csv(ruta_salida_enriquecida, index=False)

print(f"✅ Nuevo archivo maestro creado exitosamente en:\n   {ruta_salida_enriquecida}")
print("✅ Script finalizado. Toda la arquitectura está lista para el EDA.")

 💾 PASO 4: GUARDAR BASE DE DATOS ENRIQUECIDA
✅ Nuevo archivo maestro creado exitosamente en:
   /content/drive/MyDrive/dataset_master_enriquecido_1516.csv
✅ Script finalizado. Toda la arquitectura está lista para el EDA.
